In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.tools import TavilySearchResults
from langchain.memory import ConversationBufferMemory
from dotenv import load_dotenv
import os
# 🔐 Load API keys
load_dotenv(".env")
tavily_api_key = os.getenv("TAVILY_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Simple QA tool (LLMChain)
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.run,
    description="Answer the question without hallucination"
)

# Web search tool (Tavily)
search_tool = Tool(
    name="Web_Search",
    func=TavilySearchResults(max_results=3).run,
    description="Search the internet for current info"
)

tools = [qa_tool, search_tool]
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\Paul\AppData\Local\Temp\ipykernel_8416\2983859225.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
C:\Users\Paul\AppData\Local\Temp\ipykernel_8416\2983859225.py:27: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  func=TavilySearchResults(max_results=3).run,
C:\Users\Paul\AppData\Local\Temp\ipykernel_8416\2983859225.py:32: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="c

In [2]:
#1) Zero-Shot ReAct (simple + reliable default)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors = True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")

C:\Users\Paul\AppData\Local\Temp\ipykernel_8416\940164127.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
C:\Users\Paul\AppData\Local\Temp\ipykernel_8416\940164127.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = agent.run("Summarize LangChain in 2 lines, then te



> Entering new AgentExecutor chain...
Action: Simple_QA
Action Input: What is LangChain and who created it?
Observation: Here's a clear answer to your questions:

**What is LangChain?**

LangChain is an **open-source framework** designed to simplify the creation of applications powered by large language models (LLMs). It provides a set of tools, components, and interfaces that allow developers to:

*   **Connect LLMs to external data sources:** Enabling applications to retrieve relevant information (e.g., from databases, documents, web) to augment LLM responses (often called Retrieval Augmented Generation or RAG).
*   **Allow LLMs to interact with other tools:** Giving LLMs the ability to perform actions like searching the web, running code, or interacting with APIs.
*   **Manage conversational memory:** Helping LLMs remember past interactions within a conversation.
*   **Chain together complex sequences of operations:** Orchestrating multiple LLM calls and tool interactions to achie

In [3]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs), enabling them to connect with other data sources and interact with their environment. It was created by Harrison Chase.

> Finished chain.


In [4]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "Here's a 3-step study path for LangChain:\n\n1.  **Master the Fundamentals:** Begin by understanding LangChain's core components like Models (LLMs, ChatModels), Prompts, Chains, and basic data loading/processing (Document Loaders, Text Splitters). Focus on the official documentation and simple examples to grasp how these pieces connect.\n\n2.  **Build Practical Applications:** Apply your knowledge by creating foundational projects. Implement basic LLM chains for tasks like Q&A or summarization, integrate external data for Retrieval Augmented Generation (RAG) using vector stores, and experiment with simple chatbots incorporating memory.\n\n3.  **Explore Advanced Concepts & Agents:** Dive into more complex features such as Agents and Tools, enabling LLMs to interact with external environments and APIs. Learn about advanced chain types and work on a more sophisticated project, like a multi

In [5]:
from langchain.tools import StructuredTool
from langchain.agents import initialize_agent, AgentType
from pydantic import BaseModel

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)




> Entering new AgentExecutor chain...
Thought: The user wants a friendly title about LangGraph tutorials. The `TitleMaker` tool seems appropriate for this task. I will use the `TitleMaker` tool with the `topic` as "LangGraph tutorials" and the `tone` as "friendly".
Action:
```
{
  "action": "TitleMaker",
  "action_input": {
    "topic": "LangGraph tutorials",
    "tone": "friendly"
  }
}
```
Observation: Friendly Title: LangGraph tutorials in Practice
Thought:The user asked for a friendly title about LangGraph tutorials. I previously used the `TitleMaker` tool and received "LangGraph tutorials in Practice" as the output. This directly answers the user's request.
Action:
```
{
  "action": "Final Answer",
  "action_input": "Here's a friendly title for LangGraph tutorials: LangGraph tutorials in Practice"
}
```

> Finished chain.
✅ Result: Here's a friendly title for LangGraph tutorials: LangGraph tutorials in Practice


In [6]:
#5)OpenAI Functions (function-calling schema)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)
result = agent.run("Search the web for LangGraph docs and give me 3 bullets.")




> Entering new AgentExecutor chain...

Invoking: `Web_Search` with `LangGraph docs`


[{'title': 'LangGraph overview - Docs by LangChain', 'url': 'https://docs.langchain.com/oss/python/langgraph/overview', 'content': 'Trusted by companies shaping the future of agents— including Klarna, Uber, J.P. Morgan, and more— LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. LangGraph is very low-level, and focused entirely on agent orchestration. Before using LangGraph, we recommend you familiarize yourself with some of the components used to build agents, starting with models and tools. We will commonly use LangChain components throughout the documentation to integrate models and tools, but you don’t need to use LangChain to use LangGraph. If you are just getting started with agents or want a higher-level abstraction, we recommend you use LangChain’s agents that provide prebuilt architectures for common LLM and too

In [8]:
#6) OpenAI Multi-Functions (can call multiple tools in one step)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,
    verbose=True
)
result = agent.run("Answer briefly from QA; also check the web for any updates.")




> Entering new AgentExecutor chain...
I can do that. What is your question?

> Finished chain.


In [9]:
#7) Self-Ask with Search (clarifies, then searches)

search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...
No.
So the final answer is: The first LangChain release was in October 2022, and it was founded by Harrison Chase.

> Finished chain.


![Model Diagram](agenttypes1.png)

![Model Diagram](agenttype2.png)

### Deep Dive

![Model Diagram](pro1.png)

![Model Diagram](pro2.png)

![Model Diagram](pro3.png)